In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils import resample

df = pd.read_csv("bicep_curl_dataset.csv")

df["label"] = df["label"].apply(lambda x: "correct" if x == "correct" else "incorrect")

df["elbow_velocity"] = df["elbow_angle"].diff().fillna(0)
df["shoulder_movement"] = df["shoulder_angle"].diff().abs().fillna(0)
df["elbow_acceleration"] = df["elbow_velocity"].diff().fillna(0)

df_majority = df[df.label == "incorrect"]
df_minority = df[df.label == "correct"]

df_minority_upsampled = resample(
    df_minority,
    replace=True,
    n_samples=len(df_majority),
    random_state=42
)

df_balanced = pd.concat([df_majority, df_minority_upsampled])

X = df_balanced[[
    "elbow_angle",
    "shoulder_angle",
    "hip_angle",
    "elbow_velocity",
    "shoulder_movement",
    "elbow_acceleration"
]]

y = df_balanced["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.9655172413793104

Classification Report:
               precision    recall  f1-score   support

     correct       1.00      0.93      0.97        15
   incorrect       0.93      1.00      0.97        14

    accuracy                           0.97        29
   macro avg       0.97      0.97      0.97        29
weighted avg       0.97      0.97      0.97        29



In [14]:
import numpy as np

sample = np.array([[60, 18, 175, -15, 1, -2]])
prediction = model.predict(sample)

print("Prediction:", prediction[0])

Prediction: correct


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [15]:
sample = np.array([[95, 22, 175, -2, 1, 0]])
prediction = model.predict(sample)

print("Prediction:", prediction[0])

Prediction: incorrect


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [16]:
sample = np.array([[70, 40, 175, -10, 10, -5]])
prediction = model.predict(sample)

print("Prediction:", prediction[0])

Prediction: incorrect


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
